# NVIDIA Nemotron Model Reasoning Challenge: SFT Stage

This notebook documents stage 1 of the planned pipeline: supervised fine-tuning on your CoT-augmented Kaggle training data using Unsloth. It intentionally keeps `HuggingFaceTB/SmolLM2-135M` as a fast smoke-test model so we can validate the pipeline before moving to the actual competition base model.

Competition alignment notes:
- The saved adapter uses LoRA rank `32`, which matches the competition maximum.
- The training examples are formatted so the final answer appears inside `\boxed{}`, matching the evaluator's preferred extraction pattern.
- A competition-valid final submission must be trained on `unsloth/Nemotron-3-Nano-30B-A3B` and exported as `submission.zip` containing `adapter_config.json`.

The first code cell installs the exact libraries needed for this smoke-test notebook.


In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.7.1" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Mamba is supported only on torch==2.7.1. If you have newer torch versions, please wait 30 minutes!
!uv pip install --no-build-isolation mamba_ssm==2.2.5 causal_conv1d==1.5.2

## Load the Base Model

This cell imports Unsloth and loads the base model plus tokenizer. The notebook keeps `HuggingFaceTB/SmolLM2-135M` on purpose for quick validation runs; later, for the real Kaggle submission, you will switch the base model back to `unsloth/Nemotron-3-Nano-30B-A3B` without changing the overall workflow.


In [2]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit", # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

# model_name = "unsloth/Nemotron-3-Nano-30B-A3B"
model_name = "HuggingFaceTB/SmolLM2-135M"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length =  4096, # 2048, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    trust_remote_code = True,
    # unsloth_force_compile = True, # UNCOMMENT THIS FOR NEMOTRON FINETUNING
    # attn_implementation = "eager", # UNCOMMENT THIS FOR NEMOTRON FINETUNING
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

HuggingFaceTB/SmolLM2-135M does not have a padding token! Will use pad_token = <|endoftext|>.


## Define the Nemotron-Style Prompt Contract

This cell creates the system instruction and a reference chat template that mimics the reasoning format we want during SFT. The most important competition-specific detail is that every target answer is trained to end inside `\boxed{}`, because the evaluator gives priority to boxed answers when extracting predictions.


In [3]:
from unsloth.chat_templates import get_chat_template
from unsloth.chat_templates import CHAT_TEMPLATES

SYSTEM_INSTRUCTION = """
Your role as an assistant involves thoroughly exploring questions through a
systematic long thinking process before providing the final precise and
accurate solution.

Structure every response into two sections named Thought and Solution.

In the Thought section, reason inside <|begin_of_thought|> and
<|end_of_thought|>.

In the Solution section, present the final answer inside
<|begin_of_solution|> and <|end_of_solution|>.

Always place the final answer inside \\boxed{}.
"""

NEMOTRON_REFERENCE_CHAT_TEMPLATE = """
{% if messages[0]['role'] == 'system' %}
{{ 'System:\n' + messages[0]['content'].strip() + '\n\n' }}
{% set loop_messages = messages[1:] %}
{% else %}
{% set loop_messages = messages %}
{% endif %}

{% for message in loop_messages %}
{% if message['role'] == 'user' %}
{{ 'User:\n' + message['content'].strip() + '\n\n' }}
{% elif message['role'] == 'assistant' %}
{{ 'Assistant:\n' + message['content'].strip() + eos_token + '\n\n' }}
{% endif %}
{% endfor %}

{% if add_generation_prompt %}
{{ 'Assistant:\n' }}
{% endif %}
"""

CHAT_TEMPLATES["nemotron-reasoning-reference"] = (
    NEMOTRON_REFERENCE_CHAT_TEMPLATE,
    "eos_token",
    False,
    None,
)

print(list(CHAT_TEMPLATES.keys()))

tokenizer = get_chat_template(
    tokenizer,
    chat_template="nemotron-reasoning-reference",
)

if tokenizer.pad_token is None:
    print(f"Padding token getting added - {tokenizer.eos_token}")
    tokenizer.pad_token = tokenizer.eos_token


['unsloth', 'zephyr', 'chatml', 'mistral', 'llama', 'vicuna', 'vicuna_old', 'vicuna old', 'alpaca', 'gemma', 'gemma_chatml', 'gemma2', 'gemma2_chatml', 'llama-3', 'llama3', 'phi-3', 'phi-35', 'phi-3.5', 'llama-3.1', 'llama-31', 'llama-3.2', 'llama-3.3', 'llama-32', 'llama-33', 'qwen-2.5', 'qwen-25', 'qwen25', 'qwen2.5', 'phi-4', 'gemma-3', 'gemma3', 'qwen-3', 'qwen3', 'gemma-3n', 'gemma3n', 'gemma-4', 'gemma4', 'gemma-4-thinking', 'gemma4-thinking', 'gpt-oss', 'gptoss', 'qwen3-instruct', 'qwen3-thinking', 'lfm-2', 'starling', 'yi-chat', 'nemotron-reasoning-reference']


ValueError: not enough values to unpack (expected 4, got 2)

## Attach LoRA Adapters

This cell wraps the base model with trainable LoRA layers. Setting `r = 32` is important because the competition limits the submitted LoRA adapter rank to at most 32.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "in_proj", "out_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
## Load the Kaggle-Derived SFT Dataset

This notebook does not use a generic public reasoning dataset. Instead, it loads your generated file `outputs/SFT_cot_bit_manipulation_data.csv`, which is derived from the competition's `train.csv` and already contains chain-of-thought style supervision in `generated_cot`.

The next code cell validates the expected columns and optionally allows a smaller smoke-test subset.


In [ ]:
from datasets import load_dataset
from pathlib import Path

DATASET_PATH = Path("outputs/SFT_cot_bit_manipulation_data.csv")
NUM_SAMPLES = None # Set to an int for a quick smoke test.

assert DATASET_PATH.exists(), f"Missing dataset: {DATASET_PATH.resolve()}"

dataset = load_dataset(
    "csv",
    data_files = str(DATASET_PATH),
    split = "train",
)

required_columns = {"prompt", "answer", "generated_cot"}
missing_columns = required_columns.difference(dataset.column_names)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

if NUM_SAMPLES is not None:
    dataset = dataset.select(range(min(NUM_SAMPLES, len(dataset))))

print({
    "dataset_path": str(DATASET_PATH),
    "rows": len(dataset),
    "columns": dataset.column_names,
})

## Convert Each Row into a Supervised Conversation

This step cleans the text fields, chooses the final answer, removes duplicated answer strings from the reasoning trace when needed, and converts every training row into a `system -> user -> assistant` conversation. The assistant response always ends with a boxed final answer so the fine-tuning target matches the competition output format.


In [ ]:
import re

DEFAULT_THOUGHT = (
    "I will inspect the examples, infer the hidden rule, verify it against the provided "
    "transformations, and then solve the target instance carefully before giving the final boxed answer."
)

def normalize_field(value):
    if value is None:
        return ""
    text = str(value).strip()
    return "" if text.lower() == "nan" else text

def choose_final_answer(generated_answer, answer):
    for candidate in (generated_answer, answer):
        text = normalize_field(candidate)
        if text:
            return text
    return ""

def strip_final_answer_from_thought(thought, answer):
    cleaned_thought = normalize_field(thought)
    cleaned_answer = normalize_field(answer)
    if not cleaned_thought:
        return DEFAULT_THOUGHT

    if cleaned_answer:
        escaped_answer = re.escape(cleaned_answer)
        trailing_patterns = [
            rf"\n*The answer is {escaped_answer}\.?\s*$",
            rf"\n*The final answer is {escaped_answer}\.?\s*$",
            rf"\n*Answer\s*:\s*{escaped_answer}\s*$",
            rf"\n*Final answer\s*:\s*{escaped_answer}\s*$",
        ]
        for pattern in trailing_patterns:
            cleaned_thought = re.sub(pattern, "", cleaned_thought, flags = re.IGNORECASE)

    cleaned_thought = cleaned_thought.strip()
    return cleaned_thought or DEFAULT_THOUGHT

def format_assistant_response(thought, answer):
    cleaned_answer = normalize_field(answer)
    cleaned_thought = strip_final_answer_from_thought(thought, cleaned_answer)
    return f"""Thought:
<|begin_of_thought|>
{cleaned_thought}
<|end_of_thought|>

Solution:
<|begin_of_solution|>
The final answer is \\boxed{{{cleaned_answer}}}.
<|end_of_solution|>"""

def generate_conversation(examples):
    prompts = examples["prompt"]
    answers = examples["answer"]
    thoughts = examples["generated_cot"]
    generated_answers = examples.get("generated_answer", answers)
    conversations = []

    for prompt, answer, thought, generated_answer in zip(prompts, answers, thoughts, generated_answers):
        final_answer = choose_final_answer(generated_answer, answer)
        conversations.append([
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": normalize_field(prompt)},
            {"role": "assistant", "content": format_assistant_response(thought, final_answer)},
        ])

    return {"conversations": conversations}

dataset = dataset.map(generate_conversation, batched = True)

## Render the Final Training Text

This cell applies the custom chat template to each conversation and stores the resulting serialized prompt in a `text` column. That `text` field is what the SFT trainer will actually consume.


In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)

## Inspect One Serialized Example

This quick check is here to verify that one training example looks correct before we start optimization. You should see the prompt, the reasoning trace, and the boxed final answer assembled into one training string.


In [ ]:
sample_idx = min(2, len(dataset) - 1)
dataset[sample_idx]["text"]

<a name="Train"></a>
## Configure Stage-1 SFT Training

This cell builds the `SFTTrainer` for the first training stage of your pipeline. The current settings use a short `max_steps = 100` smoke test for quick iteration; for a full production run on Nemotron, you can replace that with a longer step budget or epoch-based training.


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 100,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
        output_dir = "outputs/nemotron_bit_manipulation_sft",
    ),
)

## Train Only on Assistant Tokens

This masking step tells Unsloth to compute loss only on the assistant completion, not on the user prompt. That is useful here because we want the model to learn the reasoning style and final-answer format rather than memorize the input prompt text.


In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "User:\n",
    response_part = "Assistant:\n",
)

## Verify the Full Input Sequence

This diagnostic cell prints one tokenized training example so you can confirm that the full prompt and response are present in the model input after preprocessing.


In [ ]:
sample_idx = min(2, len(trainer.train_dataset) - 1)
tokenizer.decode(trainer.train_dataset[sample_idx]["input_ids"])

## Verify the Label Masking

This second diagnostic view replaces masked positions with the padding token so you can visually confirm that only the assistant response contributes to training loss.


In [ ]:
sample_idx = min(2, len(trainer.train_dataset) - 1)
tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[sample_idx]["labels"]]
).replace(tokenizer.pad_token, " ")

## Check GPU Memory Before Training

This quick utility cell records the starting GPU memory footprint. It makes it easier to judge whether the configuration will still be practical when you later swap the smoke-test model for the full Nemotron base model.


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

## Launch Supervised Fine-Tuning

This cell starts the actual SFT run. If you need to continue from an interrupted run, you can use `trainer.train(resume_from_checkpoint = True)`.


In [ ]:
trainer_stats = trainer.train()

## Summarize Runtime and VRAM Usage

This cell reports total training time and peak reserved memory. Those numbers are especially helpful when you start sizing the later Nemotron run on stronger hardware.


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
## Smoke-Test Inference

This section performs a quick local generation check after training. For Kaggle scoring, the important behaviors are deterministic decoding and a boxed final answer; this notebook keeps the generation short because it is only meant to validate the SFT pipeline.


In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_INSTRUCTION},
    {"role": "user", "content": dataset[0]["prompt"]},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True,
)

print(f"chat template:\n\n{text}\n")

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1024,  # Short smoke test; Kaggle allows much longer generations.
    temperature = 0.0,
    top_p = 1.0,
    do_sample = False,
    repetition_penalty = 1.0,
    use_cache = True,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)


<a name="Save"></a>
## Save the LoRA Adapter

This cell saves the adapter and tokenizer to `outputs/nemotron_bit_manipulation_lora`. For Kaggle, the critical file inside that folder is `adapter_config.json`, because the evaluator expects a compatible LoRA adapter rather than a merged full model.


In [ ]:
model.save_pretrained("outputs/nemotron_bit_manipulation_lora")
tokenizer.save_pretrained("outputs/nemotron_bit_manipulation_lora")
# model.push_to_hub("your_name/nemotron_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/nemotron_lora", token = "YOUR_HF_TOKEN") # Online saving

## Package a `submission.zip` Archive

This helper cell zips the saved adapter directory into `submission.zip`. The archive layout is suitable for Kaggle-style upload, but it becomes a truly valid competition submission only after the adapter has been trained on `unsloth/Nemotron-3-Nano-30B-A3B` rather than the current smoke-test model.


In [ ]:
from pathlib import Path
import zipfile

adapter_dir = Path("outputs/nemotron_bit_manipulation_lora")
submission_zip = Path("submission.zip")

assert adapter_dir.exists(), f"Missing adapter directory: {adapter_dir.resolve()}"
assert (adapter_dir / "adapter_config.json").exists(), "Missing adapter_config.json in saved adapter directory."

with zipfile.ZipFile(submission_zip, "w", compression = zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(adapter_dir.rglob("*")):
        if file_path.is_file():
            zf.write(file_path, arcname = file_path.relative_to(adapter_dir))

print({
    "submission_zip": str(submission_zip.resolve()),
    "size_mb": round(submission_zip.stat().st_size / 1024 / 1024, 3),
})


## Reload the Saved Adapter for Local Checks

This optional block shows how to load the saved adapter back from disk for another local sanity check or as a starting point for the later RL stage.


In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "outputs/nemotron_bit_manipulation_lora",
        max_seq_length = 4096,
        load_in_4bit = True,
    )

## Optional Merged Exports

These exports are convenient for other deployment workflows, but they are not the artifact required for this Kaggle competition. The competition expects a LoRA adapter submission, not a merged 16-bit or 4-bit model.


In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("outputs/nemotron_bit_manipulation_finetune_16bit", tokenizer, save_method = "merged_16bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/nemotron_bit_manipulation_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False:
    model.save_pretrained_merged("outputs/nemotron_bit_manipulation_finetune_4bit", tokenizer, save_method = "merged_4bit")
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/nemotron_bit_manipulation_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("outputs/nemotron_bit_manipulation_lora")
    tokenizer.save_pretrained("outputs/nemotron_bit_manipulation_lora")
if False: # Pushing to HF Hub
    model.push_to_hub("HF_USERNAME/nemotron_bit_manipulation_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/nemotron_bit_manipulation_lora", token = "YOUR_HF_TOKEN")

## Optional GGUF Export

This section is only for external deployment in `llama.cpp` or similar tools. It is not part of the Kaggle submission path for the Nemotron reasoning challenge.


## Save GGUF Variants If You Need Them Later

The next code cell is optional and unrelated to the Kaggle submission itself. Keep it disabled unless you specifically want GGUF artifacts for separate local experiments.


In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("outputs/nemotron_bit_manipulation_finetune", tokenizer)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("HF_USERNAME/nemotron_bit_manipulation_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("outputs/nemotron_bit_manipulation_finetune", tokenizer, quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/nemotron_bit_manipulation_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("outputs/nemotron_bit_manipulation_finetune", tokenizer, quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/nemotron_bit_manipulation_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/nemotron_bit_manipulation_finetune",
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m"],
        token = "YOUR_HF_TOKEN",
    )

## Notebook Status and Next Step

This notebook now documents the stage-1 SFT workflow clearly from environment setup to adapter export. It is aligned with your current plan of doing SFT first and leaving GRPO/PPO for a later notebook.

Before producing a real competition submission, remember to:
1. Switch the base model from `HuggingFaceTB/SmolLM2-135M` to `unsloth/Nemotron-3-Nano-30B-A3B`.
2. Keep the LoRA rank at or below 32.
3. Preserve the boxed final-answer format.
4. Zip the final adapter into `submission.zip` for Kaggle upload.
